# Challenge 05 — de novo binders to HLA-A\*11:01 / KRAS G12D

Design and evaluation pipeline, run on a Kaggle T4.

The target is the KRAS G12D neoantigen decamer `VVVGADGVGK` presented on HLA-A\*11:01. The mutated aspartate presents 8.8 Å² of solvent-accessible surface and one of its carboxylate oxygens is entirely buried, so it cannot be read by direct contact. Everything below is built around that constraint.

Each stage writes a CSV the next one reads. Scripts live in a Kaggle dataset; the cells locate them with `find` rather than a hard-coded dataset name, because dataset slugs change and a wrong path costs a session.

| stage | script | output |
|---|---|---|
| 1 design | BoltzGen | 25 designs |
| 2 geometry | `geometric_filter.py` | `geometry.csv` |
| 3 monomer refold | `monomer_refold.py` | `monomer_refold.csv` |
| 4 counter-screen | `counterscreen3.py` | `counterscreen3.csv` |
| 5 cross-panel | `crosspan.py` | `crosspan.csv` |
| 6 complex MD | `md_gpu.py` | `md_metrics_09.csv` |
| 7 replica MD | `md_replicas.py` | `replicas.csv` |

**Session requirements:** Accelerator set to GPU, and **Internet on** — stages 4 and 5 need the MSA server.

In [ ]:
import torch
assert torch.cuda.is_available(), "no GPU - sidebar > Session options > Accelerator"
print(torch.cuda.get_device_name(0))

## 1 — Design

The target is the complete free-state trimer: HLA-A\*11:01 α1-α2 (chain A), β2-microglobulin (chain B) and the G12D decamer (chain C). Binding is directed to peptide positions 4–8.

**`not_binding: "all"` on β2-microglobulin is the load-bearing line.** A trimmed α1-α2-plus-peptide target leaves the β2m socket empty and the generative model fills it: an earlier run against the trimmed target produced 142 designs with zero peptide contact, 50 of them more than 80% identical to β2m itself. Including β2m and declaring it off-limits converts an open socket into a negative-design constraint.

In [ ]:
!pip install -q boltzgen && echo "INSTALL OK"

In [ ]:
%%writefile /kaggle/working/pmhc_g12d.yaml
entities:
  - protein:
      id: D
      sequence: 55..90

  - file:
      path: target_g12d.pdb
      include:
        - chain:
            id: A          # HLA-A*11:01 alpha1-alpha2
        - chain:
            id: B          # beta2-microglobulin
        - chain:
            id: C          # KRAS G12D peptide VVVGADGVGK
      binding_types:
        - chain:
            id: C
            binding: 4,5,6,7,8
        - chain:
            id: B
            not_binding: "all"

In [ ]:
target = !find /kaggle/input -name 'target_g12d.pdb' | head -1
!cp {target[0]} /kaggle/working/
!cd /kaggle/working && boltzgen run pmhc_g12d.yaml --output ./run \
    --protocol protein-anything \
    --steps design inverse_folding folding analysis filtering \
    --num_designs 25 --budget 20

## 2 — Interface geometry

Buried surface split between peptide and MHC, hotspot coverage, cysteine count and length. The quantity that matters is Å² buried on the **peptide**, not on the complex: a design lying along the α1/α2 helices can satisfy a co-folding score while contacting no peptide at all.

The archive is written first. BoltzGen's `fold_out_npz/` files hold the folding-step confidence arrays, and they are the only copy — if the analysis step fails, everything is recoverable from them and from nothing else.

In [ ]:
!cd /kaggle/working && zip -qr designs.zip run/intermediate_designs_inverse_folded
!pip install -q biopython
flt = !find /kaggle/input -name 'geometric_filter.py' | head -1
!python {flt[0]} /kaggle/working/run/intermediate_designs_inverse_folded \
    /kaggle/working/geometry.csv

## 3 — Monomer self-consistency

Each binder folded **alone**, target absent, with Chai-1 in single-sequence mode. A design that only holds its shape when the target templates it is both an expression risk and a false positive in the co-folding score.

Single-sequence is correct here rather than a compromise: a de novo binder of 55–80 residues has no MSA by definition. That is *not* true of the MHC, which is why stage 4 needs the MSA server.

This filter is not redundant with confidence — it demoted two designs ranked 3rd and 5th on ipSAE whose monomer pLDDT came back at 0.665 and 0.725.

In [ ]:
!pip install -q chai_lab
s = !find /kaggle/input -name 'monomer_refold.py' | head -1
!python {s[0]} /kaggle/working/mono

## 4 — Three-target counter-screen

Each design against the G12D trimer, the wild-type trimer and the empty groove, within one predictor, with the on-target fold recomputed rather than carried over from BoltzGen — a difference between two predictors is not a difference between two targets.

**An earlier attempt at this was invalid because it ran with `use_msa_server=False`.** The MHC heavy chain and β2m are natural proteins and Chai needs their alignments; without them the whole complex scored near zero, including complexes Boltz-2 had placed at ipTM 0.69. The cell below checks the MSA server is reachable before spending GPU time, because a silent failure here produces plausible-looking numbers that mean nothing.

In [ ]:
!pip install -q chai_lab
import requests
print("MSA server:", requests.get("https://api.colabfold.com/", timeout=20).status_code)

In [ ]:
s = !find /kaggle/input -name 'counterscreen3.py' | head -1
!python "{s[0]}" /kaggle/working/cs

## 5 — Cross-panel selectivity

Ten HLA-A\*11:01 ligands: the five other common codon-12 substitutions, G13D, the 9-mer register of the same neoantigen, and two unrelated EBV epitopes.

The on-target peptide is repeated as the first job with the same seed and settings, so it doubles as a check that this session reproduces the counter-screen. It did, to 1.3%.

In [ ]:
s = !find /kaggle/input -name 'crosspan.py' | head -1
!python "{s[0]}" /kaggle/working/xp

## 6 — Complex molecular dynamics

Implicit-solvent MD of the lead design's complex: does the binder keep its fold, and does it stay on the peptide rather than migrating onto the helices?

**Platform note.** The PyPI OpenMM wheel ships without the CUDA platform. Its **OpenCL** platform runs on the T4 and agrees with the reference implementation to 7×10⁻⁶, giving ~377 ns/day on this system. `md_gpu.py` tries CUDA, then OpenCL, then CPU, and aborts below 4 ns/day rather than grinding silently.

In [ ]:
!pip install -q openmm pdbfixer
import openmm.testInstallation as t; t.main()

In [ ]:
!rm -rf /kaggle/working/md_out
s = !find /kaggle/input -name 'md_gpu.py' | head -1
r = !dirname $(find /kaggle/input -name 'pmhc_g12d_09.pdb' | head -1)
!python "{s[0]}" --hours 0.5 --root "{r[0]}" --out /kaggle/working/md_out

## 7 — Replica MD on both variants

The free-state ensemble put the wild-type p4–p7 bulge at 2.15× the conformational variability of G12D, with anchor residues identical. This tests whether that survives into the **bound** complex.

Three independent seeds on each variant. The wild type is built by deleting the Asp6 side chain from the identical structure, so the two systems share backbone coordinates exactly. RMSF is measured on peptide Cα after superposing each frame on the α1-α2 groove, so it reports peptide motion relative to the groove rather than tumbling of the assembly.

**The anchor residues are the control.** They should be equally rigid in both variants; if they are not, the measurement is reporting something other than bulge flexibility.

In [ ]:
!pip install -q openmm pdbfixer
!rm -rf /kaggle/working/replicas
s = !find /kaggle/input -name 'md_replicas.py' | head -1
!python "{s[0]}" --hours 0.9 --root /kaggle/input --out /kaggle/working/replicas

## Collect

Kaggle sessions end without warning and take `/kaggle/working` with them. Print the small tables into the notebook output as well as downloading them — console scrollback has recovered more than one run here.

In [ ]:
import glob
for f in sorted(glob.glob("/kaggle/working/**/*.csv", recursive=True)):
    print("=" * 70, "\n", f, "\n", open(f).read(), sep="")